# LinkedIn CEO Finder


In [ ]:
!pip install -q curl-cffi selectolax gspread python-dotenv


## THIẾT LẬP CẤU HÌNH

In [ ]:
import os
import sys
import re
import time
import urllib.parse
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import gspread
from curl_cffi import requests
from selectolax.parser import HTMLParser
from dotenv import load_dotenv

load_dotenv()

# 1. Tự động nhận diện Google Colab
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

# 2. Cấu hình tối giản Google Sheet & Proxy
DEFAULT_SHEET_ID = os.getenv('SHEET_CEO', '')  # Điền Sheet ID nếu không dùng .env
DEFAULT_WORKSHEET_NAME = os.getenv('SHEET_CEO_WORKSHEET', 'sheet1')
DEFAULT_SERVICE_ACCOUNT = os.getenv('GOOGLE_APPLICATION_CREDENTIALS', 'service_account.json')
DEFAULT_PROXY = None if IN_COLAB else os.getenv('DEFAULT_BACKBONE_PROXY', None)

# 3. Cấu hình danh mục quét website công ty
LEADERSHIP_PATHS = [
    '/leadership', '/team', '/about-us', '/about', 
    '/our-team', '/board-of-directors', '/ban-lanh-dao', 
    '/gioi-thieu', '/people', '/management'
]
CEO_KEYWORDS = ['ceo', 'chief executive', 'founder', 'co-founder', 'tổng giám đốc', 'giám đốc điều hành', 'chủ tịch']


## KẾT NỐI GG SHEET VÀ OAUTH/SA

In [ ]:
def format_sheet(ws):
    """Làm đẹp bảng dữ liệu: Đóng băng tiêu đề, tô màu header navy, căn lề và chuẩn hóa kích thước 7 cột."""
    try:
        ws.freeze(rows=1)
        ws.format('A1:G1', {
            'backgroundColor': {'red': 0.12, 'green': 0.31, 'blue': 0.47},
            'horizontalAlignment': 'CENTER',
            'verticalAlignment': 'MIDDLE',
            'textFormat': {'foregroundColor': {'red': 1.0, 'green': 1.0, 'blue': 1.0}, 'bold': True, 'fontSize': 10}
        })
        last_row = len(ws.col_values(1))
        if last_row > 1:
            # Căn giữa cột STT (A) và Ngày Cập Nhật (G), căn giữa dọc toàn bảng
            ws.format(f'A2:A{last_row}', {'horizontalAlignment': 'CENTER', 'verticalAlignment': 'MIDDLE'})
            ws.format(f'G2:G{last_row}', {'horizontalAlignment': 'CENTER', 'verticalAlignment': 'MIDDLE'})
            ws.format(f'B2:F{last_row}', {'verticalAlignment': 'MIDDLE'})

        # STT (55px) | CEO Name (200px) | Role (220px) | Keyword (120px) | Company (200px) | LinkedIn (320px) | Ngày (160px)
        col_widths = [55, 200, 220, 120, 200, 320, 160]
        requests = [{
            'updateDimensionProperties': {
                'range': {'sheetId': ws.id, 'dimension': 'COLUMNS', 'startIndex': i, 'endIndex': i + 1},
                'properties': {'pixelSize': w},
                'fields': 'pixelSize'
            }
        } for i, w in enumerate(col_widths)]
        ws.spreadsheet.batch_update({'requests': requests})
    except Exception as e:
        print(f"⚠️ Lưu ý khi format sheet: {e}")

def sort_and_renumber_sheet(ws):
    """Sắp xếp toàn bộ dữ liệu theo Ngày Cập Nhật mới nhất lên đầu và đánh lại STT chuẩn."""
    try:
        # ponytail: Đọc và sắp xếp dữ liệu 1 lần duy nhất
        all_vals = ws.get_all_values()
        if len(all_vals) <= 2:
            return
        data_rows = [r for r in all_vals[1:] if any(r)]
        # r[6] là Ngày Cập Nhật trong bảng 7 cột (mới nhất xếp trên cùng)
        data_rows.sort(key=lambda r: str(r[6] if len(r) > 6 else '') or '', reverse=True)
        for idx, r in enumerate(data_rows, start=1):
            r[0] = str(idx)
        ws.update(values=data_rows, range_name=f'A2:G{len(data_rows) + 1}')
    except Exception as e:
        print(f"⚠️ Lưu ý khi sắp xếp sheet: {e}")

def get_existing_sheet_ceos(ws):
    """Lấy danh sách định danh (LinkedIn URL, slug, tên) của CEO đã có trong Google Sheet."""
    if not ws:
        return set()
    try:
        all_vals = ws.get_all_values()
        if len(all_vals) <= 1:
            return set()

        headers = [h.strip().lower() for h in all_vals[0]]
        li_idx = next((i for i, h in enumerate(headers) if 'linkedin' in h), -1)
        name_idx = next((i for i, h in enumerate(headers) if 'name' in h or 'tên' in h or 'ceo' in h), -1)

        seen = set()
        for row in all_vals[1:]:
            if li_idx != -1 and len(row) > li_idx and row[li_idx].strip():
                clean_url = row[li_idx].strip().split('?')[0].rstrip('/').lower()
                seen.add(clean_url)
                m = re.search(r'linkedin\.com/in/([^/]+)', clean_url, re.I)
                if m:
                    seen.add(m.group(1).lower())
            if name_idx != -1 and len(row) > name_idx and row[name_idx].strip():
                seen.add(row[name_idx].strip().lower())
        return seen
    except Exception as e:
        print(f"⚠️ Lỗi đọc CEO cũ từ Sheet: {e}")
        return set()

def connect_google_sheet(service_account_path=DEFAULT_SERVICE_ACCOUNT, sheet_id=DEFAULT_SHEET_ID, worksheet_name=DEFAULT_WORKSHEET_NAME):
    """Kết nối Google Sheet với cơ chế hỗ trợ Colab OAuth & fallback Service Account."""
    sid = sheet_id or os.getenv('SHEET_CEO')
    if not sid:
        print("⚠️ Chưa cấu hình sheet_id hoặc biến môi trường SHEET_CEO.")
        return None

    ws_name = worksheet_name or DEFAULT_WORKSHEET_NAME or 'sheet1'
    sa_path = service_account_path or os.getenv('GOOGLE_APPLICATION_CREDENTIALS', 'service_account.json')

    print("⏳ Xác thực Google Client...")
    gc = None
    if IN_COLAB:
        try:
            from google.colab import auth
            from google.auth import default
            auth.authenticate_user()
            creds, _ = default()
            gc = gspread.authorize(creds)
            print("✅ Colab OAuth: Thành công.")
        except Exception:
            pass

    if not gc:
        if not os.path.exists(sa_path):
            print(f"❌ Không tìm thấy file credentials tại '{sa_path}'.")
            return None
        try:
            gc = gspread.service_account(filename=sa_path)
            print(f"✅ Service Account: Thành công ({sa_path}).")
        except Exception as e:
            print(f"❌ Lỗi xác thực Service Account: {e}")
            return None

    try:
        sh = gc.open_by_key(sid)
        print(f"✅ Mở Spreadsheet: '{sh.title}'")
    except Exception as e:
        print(f"❌ Lỗi mở Spreadsheet ID '{sid[:6]}...': {e}")
        return None

    try:
        try:
            ws = sh.worksheet(ws_name)
        except gspread.WorksheetNotFound:
            ws = sh.add_worksheet(title=ws_name, rows=100, cols=10)
            print(f"✅ Đã tạo tab mới: '{ws_name}'")

        expected = ['STT', 'CEO Name', 'Role / Title', 'Keyword', 'Company', 'LinkedIn URL', 'Ngày Cập Nhật']
        all_vals = ws.get_all_values()
        if not all_vals:
            ws.update(values=[expected], range_name='A1')
            print("✅ Đã tạo tiêu đề (Headers) mới.")
        elif all_vals[0] != expected:
            print("🔄 Chuẩn hóa bảng sang cấu trúc 7 cột gọn gàng...")
            migrated = []
            for i, r in enumerate(all_vals[1:], start=1):
                if not any(r):
                    continue
                name = r[1] if len(r) > 1 else ''
                role = r[2] if len(r) > 2 else ''
                kw = r[3] if len(r) > 3 else ''
                comp = r[4] if len(r) > 4 else ''
                li = r[5] if len(r) > 5 else ''
                date_val = r[8] if len(r) >= 9 else (r[6] if len(r) >= 7 and '-' in r[6] else datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
                migrated.append([str(i), name, role, kw, comp, li, date_val])
            ws.clear()
            ws.update(values=[expected] + migrated, range_name=f'A1:G{len(migrated) + 1}')
            print(f"✅ Đã đồng bộ {len(migrated)} dòng dữ liệu sang cấu trúc 7 cột.")

        sort_and_renumber_sheet(ws)
        format_sheet(ws)
        print(f"✅ Worksheet '{ws.title}' sẵn sàng ({ws.row_count} hàng, {ws.col_count} cột).")
        return ws
    except Exception as e:
        print(f"❌ Lỗi thao tác Worksheet '{ws_name}': {e}")
        return None

print("=" * 60)
print("🔍 KIỂM TRA KẾT NỐI GOOGLE SHEET & LÀM ĐẸP TABLE")
print("=" * 60)
test_ws = connect_google_sheet()
if test_ws:
    print(f"\n🎉 KẾT QUẢ: Sẵn sàng sử dụng tab '{test_ws.title}'!\n")
else:
    print("\n⚠️ KẾT QUẢ: Chưa kết nối được. Vui lòng kiểm tra SHEET_CEO hoặc credentials.\n")
print("=" * 60)


## Tiện Ích Mạng & Giải Mã URL


In [ ]:
def http_get(url, proxy=DEFAULT_PROXY, timeout=12, retries=2):
    """Gửi HTTP GET request với curl_cffi Chrome impersonation (có tự động thử lại nếu nghẽn mạng)."""
    for attempt in range(retries):
        try:
            r = requests.get(url, proxy=None if IN_COLAB else proxy, impersonate='chrome120', timeout=timeout)
            if r.status_code == 200:
                return HTMLParser(r.text)
            time.sleep(0.4)
        except Exception:
            time.sleep(0.4)
    return None

def unquote_yahoo_url(raw_href):
    """Giải mã link thực từ URL redirect trung gian của Yahoo (tham số RU=)."""
    m = re.search(r'RU=([^/&?]+)', raw_href, re.I)
    target = urllib.parse.unquote(m.group(1)) if m else raw_href
    return target.split('?')[0].rstrip('/')

print("=" * 60)
print("🔍 TIỆN ÍCH MẠNG & GIẢI MÃ URL")
print("=" * 60)
sample_yahoo_url = "https://r.search.yahoo.com/_ylt=Awr/RU=https%3a%2f%2fvn.linkedin.com%2fin%2fexample-ceo/RK=2/RS=123"
clean_url = unquote_yahoo_url(sample_yahoo_url)
assert clean_url == "https://vn.linkedin.com/in/example-ceo", f"Lỗi decode: {clean_url}"
print(f"✅ unquote_yahoo_url OK: {clean_url}")

test_tree = http_get("https://example.com", timeout=8)
if test_tree and test_tree.css_first("h1"):
    print(f"✅ http_get OK: '{test_tree.css_first('h1').text()}' từ example.com")
else:
    print("⚠️ http_get cảnh báo: Không phản hồi (kiểm tra internet hoặc proxy).")
print("=" * 60)


## Chuẩn Hóa Dữ Liệu Lead & Bóc Tách


In [ ]:
def is_leadership_role(role_text):
    """Kiểm tra role có phải là chức danh lãnh đạo cấp cao / CEO / Founder hợp lệ hay không."""
    if not role_text:
        return False
    r_lower = role_text.lower()
    # Loại bỏ ngay nếu role chứa từ khóa quá khứ hoặc vị trí không phải lãnh đạo
    bad_prefixes = ['ex-', 'ex ', 'former', 'cựu', 'previously', 'past ', 'advisor', 'investor', 'intern']
    if any(b in r_lower for b in bad_prefixes):
        return False
    
    # Danh sách chức danh lãnh đạo hợp lệ
    valid_roles = [
        'ceo', 'chief executive', 'founder', 'co-founder', 'cofounder',
        'tổng giám đốc', 'giám đốc điều hành', 'chủ tịch', 'president', 'managing director'
    ]
    return any(v in r_lower for v in valid_roles)

def make_lead(name, role, keyword, company, linkedin='', updated_at=None):
    """Chuẩn hóa đối tượng Lead (đã loại bỏ Twitter & Facebook)."""
    return {
        'name': name.strip(),
        'role': role.strip(),
        'keyword': keyword.strip(),
        'company': company.strip() or 'N/A',
        'linkedin_link': linkedin.strip(),
        'updated_at': updated_at or datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'url': linkedin.strip()  # Tương thích ngược
    }

def is_duplicate_ceo(clean_url, name, seen_set):
    """Kiểm tra CEO đã tồn tại trong tập hợp (qua URL, username slug hoặc họ tên)."""
    if not seen_set:
        return False
    norm_url = (clean_url or '').strip().split('?')[0].rstrip('/').lower()
    if norm_url and norm_url in seen_set:
        return True
    m = re.search(r'linkedin\.com/in/([^/]+)', norm_url, re.I)
    if m and m.group(1).lower() in seen_set:
        return True
    if name and name.strip().lower() in seen_set:
        return True
    return False

def print_lead(lead, index, prefix=''):
    """In log thông tin Lead tìm được một cách ngắn gọn, trực quan."""
    tag = f"[{prefix}] " if prefix else ""
    date_str = f" | {lead.get('updated_at', '')}" if lead.get('updated_at') else ""
    print(f" + [#{index}] {tag}{lead['name']} | {lead['role']} | {lead['company']} -> LinkedIn: {lead.get('linkedin_link') or 'N/A'}{date_str}")

def parse_linkedin_title(title, snippet=''):
    """
    Bóc tách Họ tên, Chức vụ lãnh đạo và Công ty từ title và snippet LinkedIn.
    Chỉ chấp nhận các chức danh lãnh đạo thực sự, loại bỏ các cựu nhân sự (ex-founder/ex-ceo).
    """
    clean = re.sub(r'\s*[\-|\|]\s*LinkedIn.*$', '', title, flags=re.IGNORECASE).strip()
    parts = [p.strip() for p in re.split(r'\s+[-–|]\s+', clean) if p.strip()]
    if not parts:
        return '', '', 'N/A'

    name = parts[0]
    # Bỏ qua nếu tên có vẻ là chức danh hoặc quá dài
    if len(name.split()) > 7 or any(k in name.lower() for k in ['ceo', 'founder', 'hiring']):
        name = ''

    role = ''
    company = ''

    # Duyệt qua các vế còn lại để tìm vế chứa chức danh lãnh đạo HIỆN TẠI
    for p in parts[1:]:
        p_lower = p.lower()
        if any(b in p_lower for b in ['ex-', 'ex ', 'former', 'cựu', 'previously', 'past ']):
            continue

        # Tách role @ company (ưu tiên phân tách theo @, at, of)
        split_role = re.split(r'\s*(?:@|\bat\b|\bof\b)\s*', p, maxsplit=1, flags=re.I)
        candidate_role = split_role[0].strip()
        candidate_company = split_role[1].strip() if len(split_role) == 2 else ''

        if is_leadership_role(candidate_role):
            role = candidate_role
            if candidate_company:
                company = candidate_company
            break

    # Nếu chưa tìm thấy role lãnh đạo ở các vế ghép, kiểm tra các vế đơn lẻ
    if not role:
        for p in parts[1:]:
            if is_leadership_role(p):
                role = p
                break

    # Nếu vẫn không phải chức danh lãnh đạo, loại bỏ ngay ứng viên này
    if not is_leadership_role(role):
        return name, '', 'N/A'

    # Bóc tách tên công ty từ snippet nếu tiêu đề chưa có
    if not company and snippet:
        m = re.search(r'(?:Experience:\s*([^·\n]+)|(?:at|of)\s+([A-Z0-9][a-zA-Z0-9\s\.\-]{2,30}))', snippet)
        if m:
            company = (m.group(1) or m.group(2) or '').strip()

    if company:
        company = re.split(r'\s*[•|–—]\s*', company)[0].strip()
        company = re.sub(r'\.\.\.$', '', company).strip()
    return name, role, company or 'N/A'

print("=" * 60)
print("🔍 BÓC TÁCH TITLE LINKEDIN & XÁC THỰC LÃNH ĐẠO")
print("=" * 60)
test_title = "Nguyen Van A - Chief Executive Officer @ Tech Corp | LinkedIn"
t_name, t_role, t_comp = parse_linkedin_title(test_title)
assert t_name == "Nguyen Van A", f"Tên sai: {t_name}"
assert t_role == "Chief Executive Officer", f"Chức danh sai: {t_role}"
assert t_comp == "Tech Corp", f"Công ty sai: {t_comp}"
print(f"✅ parse_linkedin_title OK: '{t_name}' | '{t_role}' | '{t_comp}'")

# Kiểm tra trường hợp chức danh không hợp lệ (như Product Lead có cựu danh xưng CEO)
bad_title = "Henry Scott-Green - Product @ OpenAI | ex Co-Founder/CEO at Context.ai"
b_name, b_role, b_comp = parse_linkedin_title(bad_title)
assert b_role == "", "Phải loại bỏ chức danh không phải lãnh đạo hiện tại!"
print("✅ Bộ lọc loại bỏ thành công cựu CEO / nhân sự không phải lãnh đạo hiện tại.")

sample_lead = make_lead(t_name, t_role, "AI", t_comp, linkedin="https://linkedin.com/in/example")
print("✅ Hiển thị mẫu Lead qua print_lead:")
print_lead(sample_lead, 1, prefix="Test")
print("=" * 60)


## Tìm Kiếm CEO & Cào Dữ Liệu


In [ ]:
def extract_ceo_from_website(domain, proxy=DEFAULT_PROXY, seen_urls=None):
    """Quét nhanh trang Leadership/About của website công ty (giới hạn 3s/path)."""
    domain_clean = re.sub(r'^https?://(www\.)?', '', domain).strip('/').split('/')[0]
    if not domain_clean:
        return None

    quick_paths = ['/about', '/leadership', '/team']
    for path in quick_paths:
        url = f"https://{domain_clean}{path}"
        tree = http_get(url, proxy=proxy, timeout=3)
        if not tree:
            continue

        for a in tree.css('a[href*="linkedin.com/in/"]'):
            m = re.search(r'(https?://[a-z]{0,3}\.?linkedin\.com/in/[a-zA-Z0-9_\-%]+)', a.attributes.get('href', ''), re.I)
            if not m:
                continue
            clean_url = m.group(1).split('?')[0].rstrip('/')

            parent = a.parent
            for _ in range(3):
                parent = parent.parent if parent and parent.parent else parent

            context = parent.text(separator=' ', strip=True) if parent else ''
            text_lower = context.lower()
            for kw in CEO_KEYWORDS:
                if kw in text_lower:
                    lines = [l.strip() for l in context.splitlines() if l.strip()]
                    fallback_name = clean_url.split('/in/')[-1].replace('-', ' ').title()
                    name = lines[0] if lines and len(lines[0]) <= 50 else fallback_name
                    if is_duplicate_ceo(clean_url, name, seen_urls):
                        continue
                    return make_lead(name, kw.upper(), f'Domain: {domain_clean}', domain_clean, linkedin=clean_url)
    return None

def search_ceo_for_domain(domain, proxy=DEFAULT_PROXY, seen_urls=None):
    """
    THUẬT TOÁN TÌM CEO TÊN MIỀN ĐA TẦNG (Multi-Tier Executive Resolution):
    Giải quyết triệt để lỗi sai CEO các tập đoàn lớn (Google, Microsoft, Amazon, Nvidia, AMD...)
    Tầng 1: Executive Search - Xác thực danh tính CEO thực tế từ tri thức toàn cầu (tránh cựu CEO, CEO chi nhánh).
    Tầng 2: Targeted LinkedIn Lookup - Truy xuất chính xác URL LinkedIn của vị CEO đó.
    Tầng 3: Direct LinkedIn Query - Fallback cho các công ty vừa và nhỏ / chuyên biệt (như JetBrains).
    """
    dom_clean = re.sub(r'^https?://(www\.)?', '', domain).strip('/').split('/')[0]
    raw_brand = dom_clean.split('.')[0].title()
    brand_map = {
        'Google': 'Google', 'Microsoft': 'Microsoft', 'Nvidia': 'NVIDIA',
        'Amd': 'AMD', 'Amazon': 'Amazon', 'Jetbrains': 'JetBrains',
        'Apple': 'Apple', 'Meta': 'Meta', 'Openai': 'OpenAI', 'Vng': 'VNG', 'Fpt': 'FPT'
    }
    brand_name = brand_map.get(raw_brand, raw_brand)

    # TẦNG 1: Truy vấn danh tính CEO thực tế từ cơ sở dữ liệu tri thức toàn cầu
    ceo_name = None
    q_exec = f'"current CEO of {brand_name}" OR "CEO of {brand_name}" OR "Chief Executive Officer of {brand_name}"'
    tree_exec = http_get(f"https://search.yahoo.com/search?p={urllib.parse.quote(q_exec)}", proxy=proxy, timeout=10)
    if tree_exec:
        for li in tree_exec.css('div#web ol > li')[:4]:
            h3 = li.css_first('h3')
            desc = li.css_first('div.compText') or li.css_first('p')
            title_text = h3.text(strip=True) if h3 else ''
            snippet = desc.text(strip=True) if desc else ''
            full_text = f"{title_text}. {snippet}"

            # Tiêu đề Wikipedia cá nhân (vd: 'Sundar Pichai - Wikipedia', 'Satya Nadella - Wikipedia')
            m_wiki = re.search(r'^([A-ZÀ-Ỹ][a-zA-ZÀ-ỹ\.\s]{2,30})\s*[-–|]\s*Wikipedia', title_text)
            if m_wiki and not any(k in m_wiki.group(1).lower() for k in ['list', 'history', 'criticism', brand_name.lower()]):
                ceo_name = m_wiki.group(1).strip()
                break

            # '[Tên] is [the] President and CEO of [Thương hiệu]'
            m_ceo = re.search(r'([A-ZÀ-Ỹ][a-zA-ZÀ-ỹ\.\s]{2,30})\s+(?:is\s+(?:the\s+)?(?:President\s+and\s+)?(?:CEO|Chief Executive Officer)\s+of|là\s+Tổng\s+Giám\s+Đốc\s+của)\s+' + re.escape(brand_name), full_text, re.I)
            if m_ceo:
                candidate = m_ceo.group(1).strip()
                if len(candidate.split()) <= 4 and not any(w in candidate.lower() for w in ['who', 'he', 'she', 'former', 'why']):
                    ceo_name = candidate
                    break

            # '[Thương hiệu] CEO, [Tên]'
            m_ceo2 = re.search(re.escape(brand_name) + r'\s+CEO[,\s]+([A-ZÀ-Ỹ][a-zà-ỹ]+(?:\s+[A-ZÀ-Ỹ][a-zà-ỹ]+){1,3})', full_text, re.I)
            if m_ceo2:
                ceo_name = m_ceo2.group(1).strip()
                break

    # Nếu Tầng 1 đã xác thực được danh tính CEO, tìm đúng LinkedIn URL của người đó
    if ceo_name:
        q_li = f'site:linkedin.com/in/ "{ceo_name}" "{brand_name}"'
        tree_li = http_get(f"https://search.yahoo.com/search?p={urllib.parse.quote(q_li)}", proxy=proxy, timeout=10)
        if tree_li:
            for li in tree_li.css('div#web ol > li')[:3]:
                link_tag = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
                if not link_tag:
                    continue
                clean_url = unquote_yahoo_url(link_tag.attributes.get('href', ''))
                if 'linkedin.com/in/' in clean_url:
                    return make_lead(ceo_name, 'CEO', f'Domain: {dom_clean}', brand_name, linkedin=clean_url)

    # Direct LinkedIn Query (Fallback cho công ty vừa/nhỏ, chuyên biệt)
    q_dir = f'site:linkedin.com/in/ ("CEO" OR "Chief Executive Officer" OR "Founder" OR "Tổng Giám Đốc") ("{dom_clean}" OR "{brand_name}") -intitle:former -intitle:ex -intitle:"ex-" -intitle:assistant -intitle:intern'
    tree_dir = http_get(f"https://search.yahoo.com/search?p={urllib.parse.quote(q_dir)}", proxy=proxy, timeout=12)
    if tree_dir:
        for li in tree_dir.css('div#web ol > li')[:5]:
            h3 = li.css_first('h3')
            link_tag = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
            if not h3 or not link_tag:
                continue

            title_text = h3.text(strip=True)
            clean_url = unquote_yahoo_url(link_tag.attributes.get('href', ''))
            if 'linkedin.com/in/' not in clean_url:
                continue

            desc_tag = li.css_first('div.compText') or li.css_first('p')
            snippet = desc_tag.text(strip=True) if desc_tag else ''

            name, role, company = parse_linkedin_title(title_text, snippet=snippet)
            if not name or not is_leadership_role(role):
                continue

            # Đảm bảo title hoặc snippet có liên quan đến thương hiệu
            if brand_name.lower() not in title_text.lower() and dom_clean.lower() not in title_text.lower():
                continue

            comp_final = company if company not in ('', 'N/A') else brand_name
            return make_lead(name, role, f'Domain: {dom_clean}', comp_final, linkedin=clean_url)

    return None

def search_ceos_by_keyword(keyword, target_count=10, proxy=DEFAULT_PROXY, seen_urls=None, skip_empty_company=False, max_pages=10):
    """Tìm kiếm CEO theo từ khóa qua Search Engine (xác thực chức danh nghiêm ngặt, tự động bỏ qua CEO cũ)."""
    seen_urls = seen_urls if seen_urls is not None else set()
    leads = []
    page = 0

    while len(leads) < target_count and page < max_pages:
        offset = page * 7 + 1
        page += 1
        kw_parts = ' '.join(f'"{w}"' for w in keyword.strip().split() if len(w) > 1) if ' ' in keyword else f'"{keyword}"'
        query = f'site:linkedin.com/in/ ("CEO" OR "Chief Executive Officer" OR "Founder") {kw_parts} -intitle:former -intitle:ex -intitle:"ex-" -intitle:assistant -intitle:intern'
        url = f"https://search.yahoo.com/search?p={urllib.parse.quote(query)}&b={offset}"
        tree = http_get(url, proxy=proxy, timeout=12)
        if not tree:
            break

        results = tree.css('div#web ol > li')
        if not results:
            break

        for li in results:
            h3 = li.css_first('h3')
            link_tag = li.css_first('a[href*="RU="]') or li.css_first('a[href*="linkedin.com"]')
            if not h3 or not link_tag:
                continue

            title_text = h3.text(strip=True)
            clean_url = unquote_yahoo_url(link_tag.attributes.get('href', ''))
            if 'linkedin.com/in/' not in clean_url or clean_url in seen_urls:
                continue

            desc_tag = li.css_first('div.compText') or li.css_first('p')
            snippet = desc_tag.text(strip=True) if desc_tag else ''

            name, role, company = parse_linkedin_title(title_text, snippet=snippet)
            
            if not name or len(name) <= 2 or not is_leadership_role(role):
                continue
            if skip_empty_company and company in ('', 'N/A'):
                continue

            if is_duplicate_ceo(clean_url, name, seen_urls):
                print(f"  ⏭️ Bỏ qua CEO đã có trong Sheet: {name} ({company}) -> Tiếp tục tìm CEO khác...")
                continue

            seen_urls.add(clean_url)
            m = re.search(r'linkedin\.com/in/([^/]+)', clean_url, re.I)
            if m:
                seen_urls.add(m.group(1).lower())
            seen_urls.add(name.strip().lower())

            lead = make_lead(name, role, keyword, company, linkedin=clean_url)
            leads.append(lead)
            print_lead(lead, len(leads))
            if len(leads) >= target_count:
                break
        time.sleep(0.5)
    return leads

print("=" * 60)
print("🔍 THỬ NGHIỆM TÌM NHANH 1 CEO (KEYWORD 'AI')")
print("=" * 60)
test_leads = search_ceos_by_keyword("AI", target_count=1)
if test_leads:
    print(f"\n🎉 Test tìm kiếm CEO thành công: Tìm thấy {len(test_leads)} lead!")
else:
    print("\n⚠️ Chưa lấy được lead mẫu (có thể do mạng chậm hoặc Search Engine giới hạn tạm thời).")
print("=" * 60)


## Xuất Dữ Liệu Google Sheet


In [ ]:
def save_leads_to_sheet(leads, worksheet=None, sheet_id=DEFAULT_SHEET_ID, worksheet_name=DEFAULT_WORKSHEET_NAME, service_account_path=DEFAULT_SERVICE_ACCOUNT):
    """Lưu danh sách leads vào Google Sheet: Ngày cập nhật gần nhất luôn xếp trên cùng, STT liên tục (7 cột)."""
    if not leads:
        print("⚠️ Không có dữ liệu để lưu.")
        return False

    ws = worksheet or connect_google_sheet(service_account_path=service_account_path, sheet_id=sheet_id, worksheet_name=worksheet_name)
    if not ws:
        print("❌ Không thể kết nối Google Sheet để lưu.")
        return False

    try:
        all_vals = ws.get_all_values()
        cols = ['name', 'role', 'keyword', 'company', 'linkedin_link']

        # 1. Dữ liệu mới tìm được
        new_data = []
        for l in leads:
            updated_at = l.get('updated_at') or datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            new_data.append([l.get(k, '') for k in cols] + [updated_at])

        # 2. Dữ liệu hiện có trong Sheet (bỏ cột STT cũ)
        existing_data = []
        for r in all_vals[1:]:
            if any(r):
                # Bảng 7 cột: [STT, Name, Role, Keyword, Company, LinkedIn, Ngày]
                row_info = list(r[1:7])
                while len(row_info) < 6:
                    row_info.append('')
                existing_data.append(row_info)

        # 3. Gộp và sắp xếp theo Ngày Cập Nhật mới nhất lên trên cùng (descending)
        combined = new_data + existing_data
        combined.sort(key=lambda r: str(r[-1] or ''), reverse=True)

        # 4. Gắn STT 1 -> N từ trên xuống dưới
        final_rows = [[i] + r for i, r in enumerate(combined, start=1)]

        # 5. Cập nhật vào Sheet và áp dụng định dạng 7 cột
        ws.update(values=final_rows, range_name=f'A2:G{len(final_rows) + 1}')
        format_sheet(ws)
        print(f"✅ Đã lưu {len(leads)} CEO mới (xếp mới nhất lên trên cùng, tổng cộng {len(final_rows)} dòng).")
        return True
    except Exception as e:
        print(f"❌ Lỗi ghi dữ liệu vào Sheet: {e}")
        return False

print("=" * 60)
print("🔍 KIỂM TRA HÀM LƯU SHEET (SAVE LEADS)")
print("=" * 60)
if 'test_ws' in locals() and test_ws is not None:
    print(f"✅ Worksheet '{test_ws.title}' sẵn sàng nhận dữ liệu từ save_leads_to_sheet().")
else:
    print("ℹ️ Chưa có kết nối Worksheet hoạt động. Hàm save_leads_to_sheet đã sẵn sàng.")
print("=" * 60)


## HÀM ĐIỀU PHỐI CHÍNH

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def find_ceos_by_domains(
    company_domains, target_count=None, save_to_sheet=True,
    max_workers=5, sheet_id=DEFAULT_SHEET_ID, worksheet_name=DEFAULT_WORKSHEET_NAME,
    proxy=DEFAULT_PROXY, service_account_path=DEFAULT_SERVICE_ACCOUNT
):
    """
    HÀM RIÊNG BIỆT: Tìm kiếm CEO từ danh sách tên miền (website) công ty cụ thể theo cơ chế SONG SONG.
    Áp dụng thuật toán đa tầng (Multi-tier Resolution):
    - Tự động nhận diện chính xác CEO của các tập đoàn lớn (Google, Microsoft, Amazon, Nvidia, AMD...).
    - Hỗ trợ quét nhanh trang web đối với công ty vừa/nhỏ.

    Parameters:
        company_domains (list | str): Danh sách hoặc 1 tên miền (vd: ['https://www.google.com', 'nvidia.com'])
        target_count (int, optional): Số lượng CEO tối đa cần lấy (mặc định: lấy hết các domain hợp lệ)
        save_to_sheet (bool): Tự động lưu kết quả vào Google Sheet (mặc định: True)
        max_workers (int): Số luồng chạy song song đồng thời (mặc định: 5 luồng)
        sheet_id (str): ID của Google Sheet
        worksheet_name (str): Tên tab sheet cần lưu
        proxy (str): Proxy HTTP để cào dữ liệu
        service_account_path (str): Đường dẫn credentials Service Account

    Returns:
        list[dict]: Danh sách leads CEO tìm được theo tên miền
    """
    if isinstance(company_domains, str):
        company_domains = [company_domains]

    # Làm sạch danh sách domain và loại bỏ trùng lặp đầu vào
    clean_domains = []
    seen_dom_set = set()
    for d in company_domains:
        d_clean = re.sub(r'^https?://(www\.)?', '', d).strip('/').split('/')[0]
        if d_clean and d_clean.lower() not in seen_dom_set:
            seen_dom_set.add(d_clean.lower())
            clean_domains.append(d_clean)

    if not clean_domains:
        print("⚠️ Không có tên miền hợp lệ để tìm kiếm.")
        return []

    print(f"\n🌐 BẮT ĐẦU TÌM KIẾM CEO SONG SONG [Số lượng website: {len(clean_domains)} | Số luồng: {max_workers}]")
    leads, seen_urls = [], set()

    # 1. Tải trước danh sách CEO trong Google Sheet để tránh trùng lặp
    ws = None
    if save_to_sheet:
        ws = connect_google_sheet(service_account_path=service_account_path, sheet_id=sheet_id, worksheet_name=worksheet_name)
        if ws:
            sheet_ceos = get_existing_sheet_ceos(ws)
            seen_urls.update(sheet_ceos)
            if sheet_ceos:
                print(f"📋 Đã tải {len(sheet_ceos)} định danh CEO từ Sheet để bỏ qua CEO trùng lặp.")

    # 2. Định nghĩa hàm xử lý cho 1 domain với thuật toán đa tầng
    def _fetch_ceo_for_domain(idx, dom):
        # Giãn nhẹ giữa các luồng để tránh nghẽn request Yahoo
        time.sleep(idx * 0.15)
        # Bước 2.1: Thuật toán Executive Search & Targeted LinkedIn Lookup (Độ chính xác cao nhất)
        res = search_ceo_for_domain(dom, proxy=proxy, seen_urls=seen_urls)
        # Bước 2.2: Fallback quét trực tiếp website nếu chưa tìm thấy
        if not res:
            res = extract_ceo_from_website(dom, proxy=proxy, seen_urls=seen_urls)
        return dom, res

    # 3. ponytail: Chạy song song qua ThreadPoolExecutor (Standard Library, cực nhanh)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_dom = {executor.submit(_fetch_ceo_for_domain, i, dom): dom for i, dom in enumerate(clean_domains)}
        for future in as_completed(future_to_dom):
            dom = future_to_dom[future]
            try:
                domain_name, res = future.result()
                if not res:
                    print(f"  ⚠️ Chưa tìm thấy thông tin CEO phù hợp cho: {domain_name}")
                    continue

                if is_duplicate_ceo(res['linkedin_link'], res['name'], seen_urls):
                    print(f"  ⏭️ Đã có trong Sheet: {res['name']} ({domain_name}) -> Bỏ qua.")
                    continue

                # Cập nhật seen_urls
                seen_urls.add(res['linkedin_link'])
                m = re.search(r'linkedin\.com/in/([^/]+)', res['linkedin_link'], re.I)
                if m:
                    seen_urls.add(m.group(1).lower())
                seen_urls.add(res['name'].strip().lower())

                leads.append(res)
                print_lead(res, len(leads), prefix="Domain")

                if target_count and len(leads) >= target_count:
                    print(f"🎯 Đã đạt đủ số lượng mục tiêu: {target_count} CEO.")
                    executor.shutdown(wait=False, cancel_futures=True)
                    break
            except Exception as e:
                print(f"  ❌ Lỗi khi quét {dom}: {e}")

    print(f"\n🎯 Tổng cộng tìm thấy: {len(leads)} CEO từ tên miền.")

    # 4. Lưu vào Sheet nếu được yêu cầu
    if save_to_sheet and leads:
        save_leads_to_sheet(
            leads, worksheet=ws, sheet_id=sheet_id, worksheet_name=worksheet_name,
            service_account_path=service_account_path
        )

    return leads


def find_ceos_by_keyword(
    keyword='AI', target_count=10, save_to_sheet=True,
    sheet_id=DEFAULT_SHEET_ID, worksheet_name=DEFAULT_WORKSHEET_NAME,
    proxy=DEFAULT_PROXY, service_account_path=DEFAULT_SERVICE_ACCOUNT,
    skip_empty_company=False, max_pages=10
):
    """
    HÀM RIÊNG BIỆT: Tìm kiếm CEO theo từ khóa ngành nghề / vị trí (vd: 'Fintech', 'AI Vietnam').

    Parameters:
        keyword (str): Từ khóa ngành nghề / vị trí
        target_count (int): Số lượng CEO mục tiêu cần tìm
        save_to_sheet (bool): Tự động lưu kết quả vào Google Sheet (mặc định: True)
        sheet_id (str): ID của Google Sheet
        worksheet_name (str): Tên tab sheet cần lưu
        proxy (str): Proxy HTTP để cào dữ liệu
        service_account_path (str): Đường dẫn credentials Service Account
        skip_empty_company (bool): Bỏ qua nếu không trích xuất được công ty (mặc định: False)
        max_pages (int): Số trang Search Engine tối đa sẽ duyệt qua (mặc định: 10)

    Returns:
        list[dict]: Danh sách leads CEO mới tìm được theo từ khóa
    """
    print(f"\n🚀 BẮT ĐẦU TÌM KIẾM CEO THEO TỪ KHÓA [Keyword: '{keyword}' | Mục tiêu: {target_count}]")
    leads, seen_urls = [], set()

    ws = None
    if save_to_sheet:
        ws = connect_google_sheet(service_account_path=service_account_path, sheet_id=sheet_id, worksheet_name=worksheet_name)
        if ws:
            sheet_ceos = get_existing_sheet_ceos(ws)
            seen_urls.update(sheet_ceos)
            if sheet_ceos:
                print(f"📋 Đã tải {len(sheet_ceos)} định danh CEO từ Sheet để bỏ qua CEO trùng lặp.")

    # Quét qua Search Engine
    leads = search_ceos_by_keyword(
        keyword=keyword, target_count=target_count, proxy=proxy,
        seen_urls=seen_urls, skip_empty_company=skip_empty_company,
        max_pages=max_pages
    )

    print(f"\n🎯 Tổng cộng tìm thấy: {len(leads)} CEO mới.")

    # Lưu vào Sheet nếu được yêu cầu
    if save_to_sheet and leads:
        save_leads_to_sheet(
            leads, worksheet=ws, sheet_id=sheet_id, worksheet_name=worksheet_name,
            service_account_path=service_account_path
        )

    return leads


## VÍ DỤ SỬ DỤNG

In [ ]:
# ==============================================================================
# CÁCH 1: Tìm kiếm CEO từ danh sách TÊN MIỀN CỐ ĐỊNH (Có tỉ lệ đúng cao hơn nhưng không có nghĩa CEO nào cũng dùng LinkdeIn)
# Sử dụng hàm chuyên biệt: find_ceos_by_domains()
# ==============================================================================
results_domains = find_ceos_by_domains(
    company_domains=[
        'https://www.google.com',
        'https://www.microsoft.com',
        'https://www.nvidia.com',
        'https://www.amd.com',
        'https://www.jetbrains.com',
    ],
    max_workers=5,
    save_to_sheet=True
)

# ==============================================================================
# CÁCH 2: Tìm kiếm CEO theo TỪ KHÓA ngành nghề (Có thể bị sai thông tin tìm kiếm)
# Sử dụng hàm chuyên biệt: find_ceos_by_keyword())
# ==============================================================================
# results_keyword = find_ceos_by_keyword(
#     keyword='Fintech Vietnam',
#     target_count=5,
#     save_to_sheet=True
# )
